In [2]:
import pandas as pd
import numpy as np

# Load the dataframe
df = pd.read_csv('data/data_platform-strategy-list_1769496701.csv')

# Display basic info to understand structure and 'mode' columns
print(df.info())

# Check for multiple modes per symbol
# Extract symbol and mode from '调度名' (Strategy ID)
# Example: fr-XMR-USDT-loong_kc_bnfutures_mode_2 -> Symbol: XMR-USDT-FTR (already in column), Mode: 2

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 548 entries, 0 to 547
Data columns (total 36 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   调度名       548 non-null    object 
 1   币对(标准)    365 non-null    object 
 2   总收益       548 non-null    float64
 3   成交收益      548 non-null    float64
 4   资金费收益     548 non-null    float64
 5   手续费       548 non-null    float64
 6   滑点        548 non-null    float64
 7   当期资金费率    548 non-null    object 
 8   当期资金费     548 non-null    float64
 9   成交量       548 non-null    float64
 10  最大持仓      548 non-null    float64
 11  当前持仓      548 non-null    float64
 12  换手率       548 non-null    float64
 13  敞口        548 non-null    float64
 14  市场成交占比    548 non-null    float64
 15  总收益率      548 non-null    float64
 16  成交收益率     548 non-null    float64
 17  资金费收益率    548 non-null    float64
 18  滑点率       548 non-null    float64
 19  手续费率      548 non-null    float64
 20  持仓占比      548 non-null    float6

In [3]:
# 展示前几行
print(df.head())

                                      调度名         币对(标准)          总收益  \
0   fr-XMR-USDT-loong_kc_bnfutures_mode_2   XMR-USDT-FTR  7611.032268   
1     fr-H-USDT-loong_kc_bnfutures_mode_1     H-USDT-FTR  2932.063011   
2  fr-HYPE-USDT-loong_kc_bnfutures_mode_3  HYPE-USDT-FTR  2454.691588   
3   fr-BAN-USDT-loong_kc_bnfutures_mode_7   BAN-USDT-FTR  2100.831760   
4   fr-BAN-USDT-loong_kc_bnfutures_mode_4   BAN-USDT-FTR  2068.733461   

         成交收益        资金费收益        手续费          滑点              当期资金费率  \
0  430.267163  7180.765105  94.784352  -65.414920  3.6663E-04 | 0E+00   
1  -51.599093  2983.662103  17.546241 -176.404420       5E-05 | 0E+00   
2   88.437556  2366.254032  16.743835 -205.613251       5E-05 | 0E+00   
3   32.098299  2068.733461   3.941743  -32.357305  1.9641E-04 | 0E+00   
4    0.000000  2068.733461   0.000000    0.000000  1.9641E-04 | 0E+00   

        当期资金费            成交量  ...                  主机             调度最后更新时间  \
0  669.203210  284000.587054  ...  prod-stra

In [4]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

# 1. Load Data
df = pd.read_csv('data/data_platform-strategy-list_1769496701.csv')

# 2. Extract Meta Info
def extract_meta(strategy_id):
    # Pattern: algo-SYMBOL-loong_..._mode_N
    # Example: fr-XMR-USDT-loong_kc_bnfutures_mode_2
    match = re.search(r'mode_(\d+)$', strategy_id)
    mode_idx = int(match.group(1)) if match else 0
    return mode_idx

df['mode_index'] = df['调度名'].apply(extract_meta)
df['symbol'] = df['币对(标准)']

# 3. Type Conversion
df['调度最后成交时间'] = pd.to_datetime(df['调度最后成交时间'], errors='coerce')
df['调度最后更新时间'] = pd.to_datetime(df['调度最后更新时间'], errors='coerce')
# Handle numeric columns that might be strings (though info() said float, just in case)
cols_to_float = ['成交收益', '资金费收益', '手续费', '滑点', '当前持仓', '最大持仓', '敞口', '市场成交占比', '成交量']
for col in cols_to_float:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# 4. Aggregation Logic
# We need to sort by symbol and mode to identify the latest one easily
df = df.sort_values(by=['symbol', 'mode_index'])

def aggregate_strategy(x):
    # Latest mode row
    latest = x.iloc[-1]
    
    # Summations
    total_spread_pnl = x['成交收益'].sum()
    total_commission = x['手续费'].sum()
    total_slippage = x['滑点'].sum() # Negative is loss
    
    # Cumulative Funding: Take from latest mode
    # User Note: "Fund Income field directly reads the cumulative value"
    # Logic: Latest Mode Funding. 
    # Caveat: If the strategy reset its internal counter? We assume it's account level as per desc.
    total_funding = latest['资金费收益']
    
    # Total PnL (Calculated)
    # PnL = Spread + Funding + Slippage (signed) - Commission
    total_pnl = total_spread_pnl + total_funding + total_slippage - total_commission
    
    # Other metrics
    max_mode = x['mode_index'].max()
    current_pos = latest['当前持仓']
    max_pos = x['最大持仓'].max() # Max pos ever held
    last_trade = latest['调度最后成交时间']
    last_update = latest['调度最后更新时间']
    exposure = latest['敞口']
    market_share = latest['市场成交占比']
    
    # ROI Base
    # Use max_pos as "Principal". If max_pos is 0 (unlikely), use NaN
    roi_base = abs(max_pos) if abs(max_pos) > 0 else np.nan
    roi = total_pnl / roi_base if roi_base else 0
    
    return pd.Series({
        'mode_count': max_mode,
        'total_spread_pnl': total_spread_pnl,
        'total_funding': total_funding,
        'total_commission': total_commission,
        'total_slippage': total_slippage,
        'total_pnl': total_pnl,
        'current_pos': current_pos,
        'max_pos': max_pos,
        'roi': roi,
        'last_trade': last_trade,
        'last_update': last_update,
        'exposure': exposure,
        'market_share': market_share,
        'raw_slippage_cost': abs(total_slippage) # For cost efficiency calculation
    })

# Group by Symbol
df_grouped = df.groupby('symbol').apply(aggregate_strategy).reset_index()

# 5. Derived Metrics
# Cost Efficiency: (Commission + |Slippage|) / Total PnL
# Note: If Total PnL is negative or zero, this metric might be weird.
# We will display it, but interpretation depends on sign.
df_grouped['cost_ratio'] = (df_grouped['total_commission'] + df_grouped['raw_slippage_cost']) / df_grouped['total_pnl']

# Position Status
df_grouped['pos_ratio'] = (df_grouped['current_pos'].abs() / df_grouped['max_pos'].abs()).fillna(0)
def get_status(row):
    now = pd.Timestamp.now()
    # Simple time check (assuming data is somewhat recent, but "now" in VM is real-time. 
    # The data might be old. Let's rely on relative times or just values)
    if row['pos_ratio'] > 0.8:
        return 'Building/Full'
    elif row['pos_ratio'] < 0.1 and abs(row['current_pos']) > 1e-6: # Non-zero but small
        return 'Closing'
    elif abs(row['current_pos']) <= 1e-6:
        return 'Closed'
    else:
        return 'Holding'

df_grouped['status'] = df_grouped.apply(get_status, axis=1)

# PnL Source
def get_pnl_source(row):
    if row['total_pnl'] < 0:
        return 'Loss'
    elif row['total_funding'] > row['total_spread_pnl']:
        return 'Funding Driven'
    else:
        return 'Spread Driven'

df_grouped['pnl_source'] = df_grouped.apply(get_pnl_source, axis=1)

# Sort by Total PnL
df_grouped = df_grouped.sort_values(by='total_pnl', ascending=False)

In [5]:
# 聚合结果预览（完整列）
print(df_grouped.head(10))

                symbol  mode_count  total_spread_pnl  total_funding  \
156       XMR-USDT-FTR           2        430.267163    7180.765105   
59          H-USDT-FTR           1        -51.599093    2983.662103   
66       HYPE-USDT-FTR           3         88.437556    2366.254032   
76       KITE-USDT-FTR           1          2.336975    1915.807701   
82        LIT-USDT-FTR           0         26.504922    1648.215150   
36        CYS-USDT-FTR           0          1.223424    1250.941413   
52   FARTCOIN-USDT-FTR           2       -144.919986    1525.329065   
79      LIGHT-USDT-FTR           0         21.521151    1198.679684   
91       NEAR-USDT-FTR           4        801.921688     282.861125   
13      ASTER-USDT-FTR           1       1050.284466     203.019349   

     total_commission  total_slippage    total_pnl   current_pos  \
156         94.784352      -65.414920  7450.832996 -1.825282e+06   
59          17.546241     -176.404420  2738.112350 -7.311267e+05   
66          16

In [6]:
# 聚合结果预览（关键列）
print(df_grouped[['symbol', 'total_pnl', 'total_spread_pnl', 'total_funding', 'cost_ratio', 'pnl_source']].head(10))

                symbol    total_pnl  total_spread_pnl  total_funding  \
156       XMR-USDT-FTR  7450.832996        430.267163    7180.765105   
59          H-USDT-FTR  2738.112350        -51.599093    2983.662103   
66       HYPE-USDT-FTR  2232.334502         88.437556    2366.254032   
76       KITE-USDT-FTR  1917.700518          2.336975    1915.807701   
82        LIT-USDT-FTR  1596.884380         26.504922    1648.215150   
36        CYS-USDT-FTR  1249.358406          1.223424    1250.941413   
52   FARTCOIN-USDT-FTR  1147.657371       -144.919986    1525.329065   
79      LIGHT-USDT-FTR  1059.808710         21.521151    1198.679684   
91       NEAR-USDT-FTR   969.857193        801.921688     282.861125   
13      ASTER-USDT-FTR   928.279203       1050.284466     203.019349   

     cost_ratio      pnl_source  
156    0.021501  Funding Driven  
59     0.070834  Funding Driven  
66     0.099607  Funding Driven  
76     0.000232  Funding Driven  
82     0.048742  Funding Driven  
36 

In [7]:
# 盈亏构成堆叠图 (PnL Composition Stacked Bar)
# X轴: 币种 | Y轴: 金额(USDT) | 堆叠: 价差收益、资金费收益
import plotly.graph_objects as go

def pnl_stacked_bar(plot_df, title):
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=plot_df['symbol'],
        y=plot_df['total_spread_pnl'],
        name='价差收益',
        marker_color='#2ecc71',
    ))
    fig.add_trace(go.Bar(
        x=plot_df['symbol'],
        y=plot_df['total_funding'],
        name='资金费收益',
        marker_color='#3498db',
    ))
    fig.update_layout(
        barmode='stack',
        title=title,
        xaxis_title='币种 (Symbol)',
        yaxis_title='金额 (USDT)',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        height=500,
        margin=dict(b=120),
        xaxis_tickangle=-45,
    )
    return fig

# 前20赚钱
plot_winners = df_grouped.sort_values('total_pnl', ascending=False).head(20)
pnl_stacked_bar(plot_winners, '盈亏构成 · 前20赚钱 (Top 20 Profitable)').show()

# 前20亏钱
plot_losers = df_grouped.sort_values('total_pnl', ascending=True).head(20)
pnl_stacked_bar(plot_losers, '盈亏构成 · 前20亏钱 (Top 20 Loss)').show()

In [8]:
# 1. Top Winners & Losers
top_winners = df_grouped.sort_values(by='total_pnl', ascending=False).head(5)
top_losers = df_grouped.sort_values(by='total_pnl', ascending=True).head(5)

# 2. Iteration Analysis
high_freq = df_grouped[df_grouped['mode_count'] >= 5].sort_values(by='mode_count', ascending=False)

# 3. Risk Analysis
high_share = df_grouped[df_grouped['market_share'] > 0.05].sort_values(by='market_share', ascending=False)
high_exposure = df_grouped.sort_values(by='exposure', key=abs, ascending=False).head(5)

# 4. Status Analysis
closing_stage = df_grouped[df_grouped['status'] == 'Closing']
building_stage = df_grouped[df_grouped['status'] == 'Building/Full']

# 5. Stale Strategies
max_time = df_grouped['last_update'].max()
stale_threshold = max_time - pd.Timedelta(hours=1)
stale_strategies = df_grouped[df_grouped['last_update'] < stale_threshold]

# 6. Cost Efficiency Worst (positive PnL but high cost ratio)
inefficient_winners = df_grouped[(df_grouped['total_pnl'] > 100) & (df_grouped['cost_ratio'] > 0.3)].sort_values(by='cost_ratio', ascending=False).head(5)

def format_table(df, cols):
    return df[cols].copy()

In [9]:
print("=== Top Winners ===")
print(format_table(top_winners, ['symbol', 'total_pnl', 'pnl_source', 'cost_ratio']))

=== Top Winners ===
            symbol    total_pnl      pnl_source  cost_ratio
156   XMR-USDT-FTR  7450.832996  Funding Driven    0.021501
59      H-USDT-FTR  2738.112350  Funding Driven    0.070834
66   HYPE-USDT-FTR  2232.334502  Funding Driven    0.099607
76   KITE-USDT-FTR  1917.700518  Funding Driven    0.000232
82    LIT-USDT-FTR  1596.884380  Funding Driven    0.048742


In [10]:
print("=== Top Losers ===")
print(format_table(top_losers, ['symbol', 'total_pnl', 'pnl_source', 'cost_ratio', 'mode_count']))

=== Top Losers ===
              symbol    total_pnl pnl_source  cost_ratio  mode_count
129  STABLE-USDT-FTR -2689.729936       Loss   -0.340829           2
22      BNB-USDT-FTR -2279.211445       Loss   -0.287506           0
87     MERL-USDT-FTR -1959.420647       Loss   -0.683266           0
27    BULLA-USDT-FTR -1786.576227       Loss   -0.007342           3
120    SENT-USDT-FTR -1679.977032       Loss   -1.036936           1


In [11]:
print("=== High Frequency Schedules ===")
print(format_table(high_freq, ['symbol', 'mode_count', 'total_pnl']))

=== High Frequency Schedules ===
               symbol  mode_count    total_pnl
141      TUT-USDT-FTR          19   -95.055286
28      CAKE-USDT-FTR          17   -20.976650
18      BARD-USDT-FTR          13    20.525468
90   MUBARAK-USDT-FTR          12   -54.409685
4        AIO-USDT-FTR          12  -165.849638
64      HOME-USDT-FTR          11    -0.215655
150     VINE-USDT-FTR          11   -27.680330
53        FF-USDT-FTR           9  -115.855075
35     CROSS-USDT-FTR           8     2.512653
17       BAN-USDT-FTR           8   309.606827
16        B2-USDT-FTR           7   -36.068389
39      DOGS-USDT-FTR           6    11.579080
48       ETC-USDT-FTR           6   -37.780197
7        APR-USDT-FTR           5  -198.372547
10      ARIA-USDT-FTR           5  -755.563360
84       LYN-USDT-FTR           5  -268.577886
144       UB-USDT-FTR           5   382.952068
3        AIN-USDT-FTR           5    -0.460759
127      SPX-USDT-FTR           5    32.468322
124     SOLV-USDT-FTR      

In [12]:
print("=== High Market Share (>5%) ===")
print(format_table(high_share, ['symbol', 'market_share', 'total_pnl']))

=== High Market Share (>5%) ===
Empty DataFrame
Columns: [symbol, market_share, total_pnl]
Index: []


In [13]:
print("=== Stale Strategies (Last update > 1h vs Data Time) ===")
print(format_table(stale_strategies, ['symbol', 'last_update', 'status']))

=== Stale Strategies (Last update > 1h vs Data Time) ===
               symbol         last_update         status
144       UB-USDT-FTR 2026-01-25 23:28:36  Building/Full
146       US-USDT-FTR 2026-01-27 05:23:01        Closing
130     STBL-USDT-FTR 2026-01-27 00:50:33  Building/Full
143      UAI-USDT-FTR 2026-01-26 17:27:22        Closing
148   VELVET-USDT-FTR 2026-01-15 17:08:18  Building/Full
..                ...                 ...            ...
86   MELANIA-USDT-FTR 2026-01-26 01:20:12        Closing
10      ARIA-USDT-FTR 2026-01-20 22:10:44  Building/Full
121    SHELL-USDT-FTR 2026-01-25 07:29:04  Building/Full
87      MERL-USDT-FTR 2026-01-25 02:04:15  Building/Full
22       BNB-USDT-FTR 2026-01-22 11:30:20        Closing

[82 rows x 3 columns]


In [14]:
print("=== Closing Stage ===")
print(format_table(closing_stage, ['symbol', 'pos_ratio', 'total_pnl']).head())

=== Closing Stage ===
            symbol  pos_ratio   total_pnl
146    US-USDT-FTR   0.039717  126.488519
143   UAI-USDT-FTR   0.032658   74.764075
68    INJ-USDT-FTR   0.028145  -14.683996
98   ORDI-USDT-FTR   0.017259  -18.177815
123   SOL-USDT-FTR   0.032525  -20.972644


In [15]:
print("=== Cost Efficiency Worst (High Cost Ratio among Profitable) ===")
print(format_table(inefficient_winners, ['symbol', 'total_pnl', 'cost_ratio', 'total_spread_pnl', 'total_funding']))

=== Cost Efficiency Worst (High Cost Ratio among Profitable) ===
              symbol   total_pnl  cost_ratio  total_spread_pnl  total_funding
55   GIGGLE-USDT-FTR  330.009404    1.268680         21.687496     726.998204
109    PUMP-USDT-FTR  331.471194    0.900442       -174.236787     804.178508
71     IRYS-USDT-FTR  407.211363    0.459006         82.061930     512.062003
132     SUI-USDT-FTR  223.460540    0.456390        318.122571       7.323206
88      MET-USDT-FTR  178.084971    0.439522        -45.222579     301.579795
